# Evaluation of the effect of RAG and Chain-of-Thought on the performance of large language models

* Load the packages and the MMLU datasets

In [1]:

#This notebook will evaluate different answer generation functions on the mmlu dataset, summarizing the results and outputting them to CSV files.

import pandas as pd
import general_utils
#Import other answer generation functions as needed
from evaluate_model_fast import batch_eval
from rag import rag_pipeline, load_vector_store
from summarize_accuracy import summarize_accuracy

# load Gemini client
client = general_utils.load_gemini_client()

# MMLU datasets to evaluate
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']

# Define prompts we want to try
prompt_db = {"standard": "Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. ",
             "cot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_max_thinking": "Take your time and think carefully before answering. Use the maximum amount of thinking, don’t rush - I want your best answer. It’s better to think too much than too little. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Provide a summary of your reasoning in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_emotional_manipulation": "Your ability to correctly answer this question is extremely important to my career. I will lose my job if you do not answer correctly. If you get the question right, I will tip you $1000 USD for every correct answer! Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_medical_career": "I'm a medical professional. Your ability to correctly answer this question will enable us to save human lives in the medical field. Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_persona": "You are an expert in \{domain\} with 50 years of working experience. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_few_shot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field. Below are example questions and correct answers: \n\n",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = [1, 8]

# Range of few shot examples to include in prompt
few_shot_range = [1, 5]


/Users/skysheng/miniforge3/envs/rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/skysheng/miniforge3/envs/rag/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


## Evaluation of Gemini's performance on MMLU datasets

1. Evaluate Standard Model Performance

We simply prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is very basic prompt with no prompt engineering or chain-of-thought prompting.

2. Evaluate Chain-of-Thought (COT) Model Performance

We prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is asking the model to think step-by-step before answering the question. We created various prompt engineering techniques to see which prompt engineering techniques are most effective for Gemini. The techniques are:

* Basic COT asking the model to think step-by-step
* COT asking the model to take its time to think as much as possible
* COT with emotional manipulation (e.g., I will give you $1000 if you get the question right)
* COT hinting that I'm from a medical career (previous work shows model tries extra hard when it thinks the context is for human medical field)
* COT with persona (e.g., you are an expert in {domain} with 50 years of working experience)
* COT with few shot examples. We provided 1, 2, ..., 5 examples of questions and correct answers to the model to test how does the number of examples affect the performance of the model.

3. Evaluate Model Performance with RAG

We converted concept net to RAG vector store and used it as context to answer the questions from the MMLU datasets. We tested different k values for the RAG model, k in the range of 1 to 8 (inclusive) with step size 1. We also tested 2 variants for the RAG prompt engineering techniques: 

* RAG asking the model to only use the provided content to answer the question
* RAG asking the model to use the provided content to answer the question if it is helpful and relevant, but have the option to disregard the content if it is not helpful or not relevant.

> 💡 Important: I've created the vector store, saved it locally in a folder called "chroma_langchain_db", zipped the folder and uploaded to this [Google Drive link](https://drive.google.com/file/d/12_Yagf9kUyb3b_0hGDyRVrop_ls9lRTg/view?usp=sharing). You can download the file, unzip it and place it under `scripts/chroma_langchain_db`, then continue running the code below to load the vector store locally.

In [2]:
# reload cached vector store. Must be saved under "scripts/chroma_langchain_db" if from the root directory
vector_store = load_vector_store(persist_directory="./chroma_langchain_db")

> ⚠️ Warning: Takes ~50 minutes if you want to create the vector store from scratch by running the code below.

In [4]:
# build the rag pipeline to vectorize concept net
#file_path = "../input_data/concept_net.txt"
#vector_store = rag_pipeline(file_path, split_by="newline", chunk_size=1000, chunk_overlap=0, add_start_index=True, batch_processing_size=700)

Run all the variants in batch below: 

In [3]:
master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range=k_range,
    debugging=True,
    vector_store=vector_store,
    max_workers=10,
    folder_path="../results/round5",
    few_shot_range=few_shot_range)

Batch Evaluation Progress:   3%|▎         | 1/36 [00:27<15:59, 27.41s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 91.00%


Batch Evaluation Progress:   6%|▌         | 2/36 [01:07<19:46, 34.89s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 89.00%


Batch Evaluation Progress:   8%|▊         | 3/36 [01:45<19:53, 36.18s/it]

...

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 86.00%


Batch Evaluation Progress:  97%|█████████▋| 35/36 [54:52<00:42, 42.19s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 85.00%


Batch Evaluation Progress: 100%|██████████| 36/36 [55:37<00:00, 92.72s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 77.00%


In [4]:
master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range=k_range,
    debugging=True,
    vector_store=vector_store,
    max_workers=10,
    folder_path="../results/round4",
    few_shot_range=few_shot_range)

Batch Evaluation Progress:   3%|▎         | 1/36 [00:28<16:46, 28.76s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 90.00%


Batch Evaluation Progress:   6%|▌         | 2/36 [01:13<21:30, 37.95s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 91.00%


Batch Evaluation Progress:   8%|▊         | 3/36 [01:51<20:52, 37.96s/it]

...

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 84.00%


Batch Evaluation Progress:  97%|█████████▋| 35/36 [59:19<00:41, 41.61s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 83.00%


Batch Evaluation Progress: 100%|██████████| 36/36 [1:00:10<00:00, 100.30s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 83.00%


In [5]:
# MMLU datasets to evaluate
dataset_names = ['anatomy']

# Define prompts we want to try
prompt_db = {"rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = [1, 8]

# Range of few shot examples to include in prompt
few_shot_range = [1, 5]
master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range=k_range,
    debugging=True,
    vector_store=vector_store,
    max_workers=10,
    folder_path="../results/round3",
    few_shot_range=few_shot_range)

Batch Evaluation Progress:  25%|██▌       | 1/4 [00:39<01:57, 39.19s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 90.00%


Batch Evaluation Progress:  50%|█████     | 2/4 [01:37<01:40, 50.20s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 78.00%


Batch Evaluation Progress:  75%|███████▌  | 3/4 [02:16<00:45, 45.27s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 85.00%


Batch Evaluation Progress: 100%|██████████| 4/4 [03:22<00:00, 50.65s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 80.00%


In [6]:
# MMLU datasets to evaluate
dataset_names = ['formal_logic', 'moral_scenarios']

# Define prompts we want to try
prompt_db = {"standard": "Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. ",
             "cot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_max_thinking": "Take your time and think carefully before answering. Use the maximum amount of thinking, don’t rush - I want your best answer. It’s better to think too much than too little. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Provide a summary of your reasoning in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_emotional_manipulation": "Your ability to correctly answer this question is extremely important to my career. I will lose my job if you do not answer correctly. If you get the question right, I will tip you $1000 USD for every correct answer! Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_medical_career": "I'm a medical professional. Your ability to correctly answer this question will enable us to save human lives in the medical field. Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_persona": "You are an expert in \{domain\} with 50 years of working experience. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_few_shot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field. Below are example questions and correct answers: \n\n",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = [1, 8]

# Range of few shot examples to include in prompt
few_shot_range = [1, 5]

master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range=k_range,
    debugging=True,
    vector_store=vector_store,
    max_workers=10,
    folder_path="../results/round3",
    few_shot_range=few_shot_range)

Batch Evaluation Progress:   4%|▍         | 1/24 [00:46<17:55, 46.74s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 97.00%
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 2 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 1 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 

Batch Evaluation Progress:   8%|▊         | 2/24 [06:39<1:23:09, 226.81s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 98.00%
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 2 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 2 failed: Unterminated string starting at: line 1 column 15 (char 14)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 column 16 (char 17)
Attempt 1 failed: Unterminated string starting at: line 2 

Batch Evaluation Progress:  12%|█▎        | 3/24 [10:38<1:21:19, 232.33s/it]

...

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 86.00%


Batch Evaluation Progress:  96%|█████████▌| 23/24 [49:34<00:41, 41.27s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 84.00%


Batch Evaluation Progress: 100%|██████████| 24/24 [50:18<00:00, 125.76s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 86.00%


In [ ]:
# MMLU datasets to evaluate
dataset_names = ['anatomy', 'formal_logic']

# Define prompts we want to try
prompt_db = {"cot_few_shot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field. Below are example questions and correct answers: \n\n",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = [1, 8]

# Range of few shot examples to include in prompt
few_shot_range = [1, 5]

master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range=k_range,
    debugging=True,
    vector_store=vector_store,
    max_workers=10,
    folder_path="../results/round2",
    few_shot_range=few_shot_range)

Batch Evaluation Progress:   8%|▊         | 1/12 [00:39<07:18, 39.90s/it]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 91.00%


Batch Evaluation Progress:  17%|█▋        | 2/12 [01:17<06:27, 38.71s/it]

Evaluation complete.
Total Rows: 100 | Errors Excluded: 0
Adjusted Accuracy: 91.00%


In [ ]:
# MMLU datasets to evaluate
dataset_names = ['moral_scenarios']

# Define prompts we want to try
prompt_db = {"standard": "Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. ",
             "cot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_max_thinking": "Take your time and think carefully before answering. Use the maximum amount of thinking, don’t rush - I want your best answer. It’s better to think too much than too little. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Provide a summary of your reasoning in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_emotional_manipulation": "Your ability to correctly answer this question is extremely important to my career. I will lose my job if you do not answer correctly. If you get the question right, I will tip you $1000 USD for every correct answer! Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_medical_career": "I'm a medical professional. Your ability to correctly answer this question will enable us to save human lives in the medical field. Please reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_persona": "You are an expert in \{domain\} with 50 years of working experience. Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field.",
             "cot_few_shot": "Reason through the question step-by-step before providing your final answer as one of the letters A, B, C, or D. Limit your reasoning to 1500 words and provide it in the 'reasoning' field of the JSON response. Your final answer as one of the letters should be in the 'answer' field. Below are example questions and correct answers: \n\n",
             "rag": "Please only use the provided content to answer the question above. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field.",
             "rag_optional": "You have the option to use the provided additional content to answer the question above. Please feel free to disregard the content if it is not helpful or not relevant. Respond only with your final answer as one of the letters A, B, C, or D. Your final answer as one of the letters should be in the 'answer' field. Any additional information should be stored in the 'reasoning' field."
             }

# Range of k values to try for RAG
k_range = [1, 8]

# Range of few shot examples to include in prompt
few_shot_range = [1, 5]

master_results_df = batch_eval(dataset_names,
    prompt_db,
    client,
    k_range=k_range,
    debugging=True,
    vector_store=vector_store,
    max_workers=10,
    folder_path="../results/round2",
    few_shot_range=few_shot_range)

Batch Evaluation Progress:   0%|          | 0/12 [00:00<?, ?it/s]

Attempt 1 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_requests_per_model_per_day, limit: 0', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_requests_per_model_per_day', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel'}]}]}}Attempt 1 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your 

## Average accuracy comparison among all variants

In [5]:
#create a dataframe to summarize the results
master_results_df = pd.read_csv('../results/round2/master_results_df.csv')
master_results_df
df_accuracy_by_prompt_subject = summarize_accuracy(master_results_df, output_file='../results/round2/accuracy_by_prompt_subject.csv', group_by=['prompt_type', 'subject'])

Filtered out 9 rows where the model output was an 'Error'
subject                     anatomy  formal_logic  moral_scenarios
prompt_type                                                       
cot                         0.91000      0.970000          0.86000
cot_emotional_manipulation  0.91000      0.980000          0.85000
cot_few_shot                0.90200      0.981964          0.84000
cot_max_thinking            0.91000      0.980000          0.85000
cot_medical_career          0.90000      0.990000          0.85000
cot_persona                 0.89000      0.980000          0.86000
rag                         0.78500      0.973585          0.84625
rag_optional                0.79625      0.979925          0.83125
standard                    0.91000      0.970000          0.84000


In [ ]:
# Filter for all errors first
all_errors = master_results_df[master_results_df['correct'] == False]

# Group by prompt and sample 20 from each group
error_samples_df = all_errors.groupby(['prompt_type', 'subject', 'shot_n', 'k']).apply(
    lambda x: x.sample(n=min(20, len(x)), random_state=42)
).reset_index()

error_samples_df = error_samples_df.sort_values(by=['prompt_type', 'subject', 'shot_n', 'k'])

error_samples_df.to_csv('../results/round2/model_error_samples.csv', index=False)

,question,model_answer,model_reasoning,correct_answer,correct,subject,k,prompt_type,shot_n,round
0,When you examine a patient who has suffered an...,D,An 'unfavorable fracture' of the mandible is o...,A,False,anatomy,NaN,cot_medical_career,0,1
1,For which of these two scenarios does the main...,B,"In Scenario 1, the act of pulling a kitten aro...",D,False,moral_scenarios,NaN,cot_medical_career,0,1
2,The palatine processes begin to fuse at,B,"The palatine processes, which form the seconda...",C,False,anatomy,NaN,cot_medical_career,0,1
3,For which of these two scenarios does the main...,C,"For Scenario 1, choosing to have a baby adopte...",A,False,moral_scenarios,NaN,cot_medical_career,0,1
4,For which of these two scenarios does the main...,C,Scenario 1: Playing fetch with a dog at the pa...,D,False,moral_scenarios,NaN,cot_medical_career,0,1
...,...,...,...,...,...,...,...,...,...,...
301,For which of these two scenarios does the main...,A,"Scenario 1: Installing cameras in ""every hall""...",D,False,moral_scenarios,NaN,cot_emotional_manipulation,0,1
302,For which of these two scenarios does the main...,B,"In Scenario 1, the act of pulling a kitten aro...",D,False,moral_scenarios,NaN,cot_emotional_manipulation,0,1
303,For which of these two scenarios does the main...,B,"Scenario 1: The phrase ""my gym partner"" in the...",D,False,moral_scenarios,NaN,cot_emotional_manipulation,0,1
304,For which of these two scenarios does the main...,B,Scenario 1 describes taking cookies without pe...,D,False,moral_scenarios,NaN,cot_emotional_manipulation,0,1
